In [145]:
# imports
import pandas as pd
import tensorflow as tf
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from constant import DATA_DIR, RANDOM_STATE


# Viewing the Data w/ Pandas

In [146]:
# loading the data
clf_df = pd.read_csv(f'{DATA_DIR}/draft.csv')

In [147]:
# looking at what the data looks like
clf_df.head(),clf_df.dtypes

(  position  longsnapper  height  weight   draftage  bench  forty  \
 0       OL            0   81.25   322.0  22.112329     19   5.40   
 1       OL            0   81.00   317.0  24.227397     16   5.46   
 2       OL            0   80.75   324.0  22.520548     18   5.53   
 3       OL            0   80.00   335.0  23.276712     17   0.00   
 4       OL            0   80.00   320.0  25.243836     22   0.00   
 
    fortyelectric  verticaljump  broadjump  threecone  shuttle  drafted  
 0          5.430         28.50     95.000      7.470    4.660        1  
 1          5.790         26.75     96.624      8.280    5.000        1  
 2          5.540         24.50     87.500      8.180    4.900        1  
 3          5.453         28.00     96.000      8.047    4.918        1  
 4          5.280         33.50     93.000      7.500    4.610        1  ,
 position             str
 longsnapper        int64
 height           float64
 weight           float64
 draftage         float64
 bench   

In [148]:
COLS = [
        "height", "weight", "draftage", "forty","bench",
        "fortyelectric", "verticaljump", "broadjump", "threecone", "shuttle"
    ]
def get_rows_with_zero(df,cols_to_remove):
    """A function that returns the dataframe that has 0 within the columns
    that do not contain a 0 by default(i.e.,height,weight,draftage,
    bench,forty,fortyelectric,verticaljump,broadjump,threecone,shuttle.).

    """

    cols = [c for c in COLS if c not in cols_to_remove]
    # for all the rows, for all the columns, we check if it equals 0
    # then return a true for that row else false(via .any).
    mask =(df[cols] == 0).any(axis=1)
    return df[mask]

get_rows_with_zero(clf_df,[]).count(),clf_df.count()
# well this is problematic since that a significant amount of the dataset. as in,
# 556/671 of them contain a 0, meaning thats missing data. so we cant remove them.
# thus, we can take the average for all rows and simply replace them
# HOWEVER, I have no clue how to do this for bench and it seems like its just two inputs,
# I am going to remove them.


    
    

(position         556
 longsnapper      556
 height           556
 weight           556
 draftage         556
 bench            556
 forty            556
 fortyelectric    556
 verticaljump     556
 broadjump        556
 threecone        556
 shuttle          556
 drafted          556
 dtype: int64,
 position         671
 longsnapper      671
 height           671
 weight           671
 draftage         671
 bench            671
 forty            671
 fortyelectric    671
 verticaljump     671
 broadjump        671
 threecone        671
 shuttle          671
 drafted          671
 dtype: int64)

In [149]:
def handle_zero_within_cols(df):
    """This function simply takes the average of the columns that do not have 0 by default.
    Then fills the 0 with the average, with the exception of bench, which those rows were removed.
    """
    # Mask momement to return a df where bench does not equal 0
    # essentially removing bench
    df = df[df["bench"] != 0]
    for col in COLS:
        avg = df[col].mean()
        df[col] = df[col].replace(0,avg)
    return df

clf_df = handle_zero_within_cols(clf_df)
clf_df
    

,position,longsnapper,height,weight,draftage,bench,forty,fortyelectric,verticaljump,broadjump,threecone,shuttle,drafted
0,OL,0,81.250,322.0,22.112329,19,5.400000,5.430,28.50,95.000,7.470000,4.660,1
1,OL,0,81.000,317.0,24.227397,16,5.460000,5.790,26.75,96.624,8.280000,5.000,1
2,OL,0,80.750,324.0,22.520548,18,5.530000,5.540,24.50,87.500,8.180000,4.900,1
3,OL,0,80.000,335.0,23.276712,17,3.431146,5.453,28.00,96.000,8.047000,4.918,1
4,OL,0,80.000,320.0,25.243836,22,3.431146,5.280,33.50,93.000,7.500000,4.610,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
665,RB,0,67.875,189.0,23.509589,15,4.780000,4.900,34.00,112.000,6.880000,3.970,0
666,WR,0,67.875,176.0,23.298630,20,4.310000,4.540,43.50,125.000,3.534859,4.030,1
668,RB,0,67.375,182.0,25.627397,6,3.431146,4.740,27.50,89.500,7.240000,4.350,0
669,K,0,67.000,165.0,21.753425,1,3.431146,4.979,30.00,102.500,7.560000,4.500,1


In [150]:
get_rows_with_zero(clf_df,[]).count(),clf_df.count()

(position         0
 longsnapper      0
 height           0
 weight           0
 draftage         0
 bench            0
 forty            0
 fortyelectric    0
 verticaljump     0
 broadjump        0
 threecone        0
 shuttle          0
 drafted          0
 dtype: int64,
 position         646
 longsnapper      646
 height           646
 weight           646
 draftage         646
 bench            646
 forty            646
 fortyelectric    646
 verticaljump     646
 broadjump        646
 threecone        646
 shuttle          646
 drafted          646
 dtype: int64)

In [151]:
# one hot encoding the position
def position_map_to_numerical(df):
    """A function that takes in a df. Grabs all the unique str within
    the `position` col. Then convert them into numerical values.

    It does not make sense to map each position label to a numerical value
    in acsending order because then we are saying one position might be better than
    the other. This is the differeniation between categorical and ordinal, as in,
    position is categorical, thus, we need a one hot encoding.
    see<https://towardsdatascience.com/encoding-categorical-variables-in-machine-learning-5644a43c123f/>
    Thus, we want to one hot encode it.
    """

    # TODO(shenukp1) Nov 15, 2026: might need a hash map of this later
    unique_arr = df["position"].unique()
    
    # see<https://www.geeksforgeeks.org/machine-learning/one-hot-encoding-from-a-pandas-column-containing-a-list/>
    one_hot = pd.get_dummies(df['position'].apply(pd.Series).stack()).groupby(level=0).sum()
    df_one_hot = one_hot.join(df.drop('position', axis=1))
    return df_one_hot
clf_df = position_map_to_numerical(clf_df)
clf_df.head()
    

,DB,DL,K,LB,OL,QB,RB,WR,longsnapper,height,weight,draftage,bench,forty,fortyelectric,verticaljump,broadjump,threecone,shuttle,drafted
0,0,0,0,0,1,0,0,0,0,81.25,322.0,22.112329,19,5.400000,5.430,28.50,95.000,7.470,4.660,1
1,0,0,0,0,1,0,0,0,0,81.00,317.0,24.227397,16,5.460000,5.790,26.75,96.624,8.280,5.000,1
2,0,0,0,0,1,0,0,0,0,80.75,324.0,22.520548,18,5.530000,5.540,24.50,87.500,8.180,4.900,1
3,0,0,0,0,1,0,0,0,0,80.00,335.0,23.276712,17,3.431146,5.453,28.00,96.000,8.047,4.918,1
4,0,0,0,0,1,0,0,0,0,80.00,320.0,25.243836,22,3.431146,5.280,33.50,93.000,7.500,4.610,1


In [152]:
# Now we need to split this into x(data) and prediction(y) variables
X, Y = clf_df.iloc[:,:-1],clf_df.iloc[:,-1]
X,Y

(     DB  DL  K  LB  OL  QB  RB  WR  longsnapper  height  weight   draftage  \
 0     0   0  0   0   1   0   0   0            0  81.250   322.0  22.112329   
 1     0   0  0   0   1   0   0   0            0  81.000   317.0  24.227397   
 2     0   0  0   0   1   0   0   0            0  80.750   324.0  22.520548   
 3     0   0  0   0   1   0   0   0            0  80.000   335.0  23.276712   
 4     0   0  0   0   1   0   0   0            0  80.000   320.0  25.243836   
 ..   ..  .. ..  ..  ..  ..  ..  ..          ...     ...     ...        ...   
 665   0   0  0   0   0   0   1   0            0  67.875   189.0  23.509589   
 666   0   0  0   0   0   0   0   1            0  67.875   176.0  23.298630   
 668   0   0  0   0   0   0   1   0            0  67.375   182.0  25.627397   
 669   0   0  1   0   0   0   0   0            0  67.000   165.0  21.753425   
 670   0   0  0   0   0   0   1   0            0  66.875   168.0  24.550685   
 
      bench     forty  fortyelectric  verticaljump

In [153]:
# The lectures mentioned that a 80/20 split is industry standard
X_train, X_test,y_train, y_test = train_test_split(X,Y,
                                   random_state=RANDOM_STATE, 
                                   test_size=0.20)


In [154]:
X_train.shape, X_test.shape,y_train.shape,y_test.shape

((516, 19), (130, 19), (516,), (130,))

In [155]:
def save_draft_test_train(X_train, X_test,y_train, y_test):
    """A function that concatentes the X with their y
    for the specific split(i.e., train or test) and saves them
    as csv.
    """
    # see<https://stackoverflow.com/questions/69658534/append-data-to-training-dataset-after-train-test-split>
    # see<https://pandas.pydata.org/docs/reference/api/pandas.concat.html> for default axis is 0
    # but we want to stack the columns
    train_df = pd.concat([X_train, y_train], axis=1)
    test_df =  pd.concat([X_test, y_test], axis=1)
    train_df.to_csv(f"{DATA_DIR}/draft_train.csv", index=False)
    test_df.to_csv(f"{DATA_DIR}/draft_test.csv", index=False)
    
save_draft_test_train(X_train, X_test,y_train, y_test)

In [161]:
def normalize(X_train, X_test):
    """Normalizes both train and test using only X_trains mean and std
    so no information from X_test leaks into training.
    
    """
    cols_to_norm = [
        "height", "weight", "draftage", "bench", "forty",
        "fortyelectric", "verticaljump", "broadjump", "threecone", "shuttle"
    ]
    scaler = StandardScaler()
    X_train[cols_to_norm] = scaler.fit_transform(X_train[cols_to_norm]) # taken from Google AI overview
    X_test[cols_to_norm] = scaler.transform(X_test[cols_to_norm])
normalize(X_train, X_test)

In [162]:
# what the spec started out with
model = tf.keras.Sequential([
 tf.keras.layers.Dense(512, activation='relu'),
 tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam',
              loss='binary_crossentropy',# because we have a binary output
              metrics=['accuracy'])


In [163]:
model.fit(X_train, y_train, epochs=10,verbose=2)

Epoch 1/10
17/17 - 1s - 34ms/step - accuracy: 0.5969 - loss: 0.6551
Epoch 2/10
17/17 - 0s - 4ms/step - accuracy: 0.6957 - loss: 0.5678
Epoch 3/10
17/17 - 0s - 4ms/step - accuracy: 0.7442 - loss: 0.5290
Epoch 4/10
17/17 - 0s - 4ms/step - accuracy: 0.7442 - loss: 0.5097
Epoch 5/10
17/17 - 0s - 4ms/step - accuracy: 0.7519 - loss: 0.4972
Epoch 6/10
17/17 - 0s - 3ms/step - accuracy: 0.7481 - loss: 0.4894
Epoch 7/10
17/17 - 0s - 4ms/step - accuracy: 0.7636 - loss: 0.4782
Epoch 8/10
17/17 - 0s - 4ms/step - accuracy: 0.7655 - loss: 0.4744
Epoch 9/10
17/17 - 0s - 4ms/step - accuracy: 0.7733 - loss: 0.4664
Epoch 10/10
17/17 - 0s - 4ms/step - accuracy: 0.7791 - loss: 0.4594


In [164]:
print("--Evaluate model for spec --")
model_loss1, model_acc1 = model.evaluate(X_train,  y_train, verbose=2)
model_loss2, model_acc2 = model.evaluate(X_test,  y_test, verbose=2)
print(f"Train / Test Accuracy: {model_acc1*100:.1f}% / {model_acc2*100:.1f}%")

--Evaluate model for spec --
17/17 - 0s - 8ms/step - accuracy: 0.7926 - loss: 0.4508
5/5 - 0s - 9ms/step - accuracy: 0.6231 - loss: 0.6636
Train / Test Accuracy: 79.3% / 62.3%


In [211]:
# training time
# see <https://www.tensorflow.org/tutorials/keras/overfit_and_underfit>
# We need this because I am training around 500 epochs and I want to stop when
# we do not see any improvement within 50 epoch
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,                 
    restore_best_weights=True,   # apparently this takes the best model for the best val loss, see<https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping>
    verbose=1 # 
)
model = tf.keras.Sequential([
 tf.keras.layers.Dense(64, activation='relu',kernel_regularizer=tf.keras.regularizers.l2(0.001)),
 tf.keras.layers.Dropout(0.2),
 tf.keras.layers.Dense(32, activation='relu',kernel_regularizer=tf.keras.regularizers.l2(0.001)),
 tf.keras.layers.Dropout(0.2),
 tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.005),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
# see<https://www.geeksforgeeks.org/deep-learning/using-early-stopping-to-reduce-overfitting-in-neural-networks/>
# for how early stopping was implemented. however, the example on this website is not right lol.
model.fit(X_train, y_train, epochs=400,callbacks=[early_stopping],validation_split=0.2, verbose=2)

Epoch 1/400
13/13 - 1s - 77ms/step - accuracy: 0.6214 - loss: 0.6998 - val_accuracy: 0.7308 - val_loss: 0.6352
Epoch 2/400
13/13 - 0s - 5ms/step - accuracy: 0.7209 - loss: 0.6103 - val_accuracy: 0.7404 - val_loss: 0.6036
Epoch 3/400
13/13 - 0s - 5ms/step - accuracy: 0.7403 - loss: 0.5939 - val_accuracy: 0.7788 - val_loss: 0.5908
Epoch 4/400
13/13 - 0s - 6ms/step - accuracy: 0.7257 - loss: 0.5662 - val_accuracy: 0.7596 - val_loss: 0.5836
Epoch 5/400
13/13 - 0s - 7ms/step - accuracy: 0.7621 - loss: 0.5397 - val_accuracy: 0.7596 - val_loss: 0.5883
Epoch 6/400
13/13 - 0s - 7ms/step - accuracy: 0.7694 - loss: 0.5309 - val_accuracy: 0.7500 - val_loss: 0.5782
Epoch 7/400
13/13 - 0s - 8ms/step - accuracy: 0.7524 - loss: 0.5329 - val_accuracy: 0.7885 - val_loss: 0.5675
Epoch 8/400
13/13 - 0s - 8ms/step - accuracy: 0.7670 - loss: 0.5182 - val_accuracy: 0.7692 - val_loss: 0.5658
Epoch 9/400
13/13 - 0s - 8ms/step - accuracy: 0.7913 - loss: 0.5036 - val_accuracy: 0.7404 - val_loss: 0.5610
Epoch 10/

In [212]:
print("--Evaluate model for my model--")
model_loss1, model_acc1 = model.evaluate(X_train,  y_train, verbose=2)
model_loss2, model_acc2 = model.evaluate(X_test,  y_test, verbose=2)
print(f"Train / Test Accuracy: {model_acc1*100:.1f}% / {model_acc2*100:.1f}%")

--Evaluate model for my model--
17/17 - 0s - 3ms/step - accuracy: 0.8120 - loss: 0.4617
5/5 - 0s - 8ms/step - accuracy: 0.6385 - loss: 0.7868
Train / Test Accuracy: 81.2% / 63.8%


In [186]:
# checking the output of the model
print(X_test.iloc[[0]])
print(y_test.iloc[0])
prediction = model.predict(X_test.iloc[[0]])
print(prediction)

     DB  DL  K  LB  OL  QB  RB  WR  longsnapper    height    weight  draftage  \
657   0   0  0   0   0   0   1   0            0 -1.875658 -0.559418  1.165715   

       bench     forty  fortyelectric  verticaljump  broadjump  threecone  \
657  0.86905  0.400148      -0.471854      0.770619  -0.218621    0.87078   

      shuttle  
657 -0.825048  
1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
[[0.19254462]]
